# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Vinod5566773/flyrank-capstone/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Unit of Analysis + Time Window

**Lane:** Refresh / Content Opportunity Scoring

**Unit of analysis:** One content item for a client, represented by one row in the aggregated feature table.

**Time window:** The analysis compares impressions in the most recent 30 days with impressions in the preceding 30 days. Content items are included when previous-window impressions are at least 100.

**Prediction target:** `is_declining` is 1 when recent impressions are less than 80% of previous-window impressions; otherwise it is 0.

**Why this grain?** Aggregating performance into two 30-day windows helps identify content items with declining impressions and prioritize them for review. The label is a rule-based proxy, not proof that a page needs refreshing.


In [1]:
unit_of_analysis = "One content item for a client"
time_window = "Two consecutive 30-day windows"
minimum_previous_impressions = 100
target_name = "is_declining"

print("Unit of analysis:", unit_of_analysis)
print("Time window:", time_window)
print("Minimum previous-window impressions:", minimum_previous_impressions)
print("Target:", target_name)

Unit of analysis: One content item for a client
Time window: Two consecutive 30-day windows
Minimum previous-window impressions: 100
Target: is_declining


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Fields

**Features**

* `imp_prev30`
* `imp_last30`
* `clk_last30`
* `pos_last30`
* `visible_queries`
* `rare_share`
* `anon_share`
* `top_query_share`
* `kept_impressions`, when used in the relevant model

These fields represent observed performance or query-pattern signals available to the analysis.

**Label / Proxy**

* `is_declining`: a binary label defined by comparing recent impressions with 80% of previous-window impressions.

**Context**

* `client_hash_id`
* `content_hash_id`

These identifiers support grouping and client-held-out validation. They are not ordinary predictive features.

**Excluded**

* Future-window measurements unavailable at decision time.
* Fields derived directly from the target.
* Client or content identifiers as ordinary model inputs.
* Outcomes observed only after the recommendation decision.

These exclusions help reduce leakage and keep the model evaluation meaningful.


In [2]:
features_used = [
    "imp_prev30",
    "imp_last30",
    "clk_last30",
    "pos_last30",
    "visible_queries",
    "rare_share",
    "anon_share",
    "top_query_share",
]

label_field = "is_declining"

context_fields = [
    "client_hash_id",
    "content_hash_id",
]

excluded_categories = [
    "future-window measurements",
    "target-derived fields",
    "identifiers used as predictive inputs",
    "post-decision outcomes",
]

print("Features:", features_used)
print("Label:", label_field)
print("Context:", context_fields)
print("Excluded:", excluded_categories)

Features: ['imp_prev30', 'imp_last30', 'clk_last30', 'pos_last30', 'visible_queries', 'rare_share', 'anon_share', 'top_query_share']
Label: is_declining
Context: ['client_hash_id', 'content_hash_id']
Excluded: ['future-window measurements', 'target-derived fields', 'identifiers used as predictive inputs', 'post-decision outcomes']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Verification

The existing analysis produced a 30-day feature table containing 111,247 content items after applying the previous-window impression threshold.

The data contract should be verified by checking:

1. The feature-table row count.
2. Whether the required fields are present.
3. Whether each row has a defined decline label.
4. Whether the previous-window impression threshold was applied.

These checks describe the existing analysis and should not require rebuilding the full warehouse table.


In [3]:
# Lightweight contract checks using the existing feature table.
# Run only if `features` is already defined in this notebook session.

required_columns = [
    "client_hash_id",
    "content_hash_id",
    "imp_last30",
    "imp_prev30",
    "is_declining",
]

print("Expected feature-table rows:", 111247)
print("Required columns:", required_columns)
print("Data contract defined.")

Expected feature-table rows: 111247
Required columns: ['client_hash_id', 'content_hash_id', 'imp_last30', 'imp_prev30', 'is_declining']
Data contract defined.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data Limits

This analysis supports decision-making, not causal conclusions.

* A decline in impressions does not explain why performance changed.
* Different content items may have different histories and traffic levels.
* A single pair of 30-day windows may not represent seasonal behaviour.
* The target is a rule-based proxy, not a human-confirmed refresh decision.
* Model performance on a random split may overstate generalization to unseen clients.
* Even a correctly identified decline does not prove that refreshing content will improve performance.

Findings should be described as observed, measured, directional, and intended for decision support. Recommendations should be reviewed by a person before action is taken.


In [4]:
limitations = [
    "Observed decline does not explain the cause.",
    "History and traffic levels may differ across content items.",
    "Two 30-day windows may miss seasonal patterns.",
    "The target is a rule-based proxy.",
    "Performance on unseen clients must be checked.",
    "A refresh is not guaranteed to improve performance.",
]

for i, limitation in enumerate(limitations, start=1):
    print(f"{i}. {limitation}")

1. Observed decline does not explain the cause.
2. History and traffic levels may differ across content items.
3. Two 30-day windows may miss seasonal patterns.
4. The target is a rule-based proxy.
5. Performance on unseen clients must be checked.
6. A refresh is not guaranteed to improve performance.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.